In [4]:
pip install scikit-optimize


   ---------------------------------------- 0/2 [pyaml]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   ---------------------------------------- 2/2 [scikit-optimize]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
"""
=============================================================================
Walk-Forward Validation & Hyperparameter Tuning — Kalman xG State-Space Model
met Asymmetrische Pi-Rating gewogen Kalman gain
=============================================================================

Hyperparameter vector θ = (σ²_w_att, σ²_w_def, σ²_v, φ, η, λ_att, λ_def, λ_lineup)

Asymmetrische Kalman gain weging
---------------------------------
De kerngedachte: de informatiewaarde van een xG observatie voor aanval
en verdediging is asymmetrisch afhankelijk van tegenstander kwaliteit.

  w_att_h = clip(1 + λ_att · away_pi_rating_pre,  0.1, 3.0)
  w_def_a = clip(1 + λ_def · home_pi_rating_pre,  0.1, 3.0)
  w_att_a = clip(1 + λ_att · home_pi_rating_pre,  0.1, 3.0)
  w_def_h = clip(1 + λ_def · away_pi_rating_pre,  0.1, 3.0)

State updates:
  α_home += K_α_h · e_home · w_att_h   ← aanval: gewogen op uitploeg sterkte
  γ_away -= K_γ_a · e_home · w_def_a   ← verdediging: gewogen op thuisploeg sterkte
  α_away += K_α_a · e_away · w_att_a   ← aanval: gewogen op thuisploeg sterkte
  γ_home -= K_γ_h · e_away · w_def_h   ← verdediging: gewogen op uitploeg sterkte

Intuïtie:
  Liverpool (sterk) scoort 3 xG tegen Norwich (zwak):
    w_att_Liverpool = 1 + λ_att · pi_Norwich  → laag  (Norwich zwak → weinig bewijs voor aanval)
    w_def_Norwich   = 1 + λ_def · pi_Liverpool → hoog  (Liverpool sterk → veel bewijs voor verdediging)

  Liverpool (sterk) scoort 3 xG tegen Man City (sterk):
    w_att_Liverpool = 1 + λ_att · pi_ManCity   → hoog  (ManCity sterk → veel bewijs voor aanval)
    w_def_ManCity   = 1 + λ_def · pi_Liverpool → hoog  (Liverpool sterk → veel bewijs voor verdediging)

Vergelijking Pi-rating specificatie:
  A) Algemeen  : gebruikt home_pi_rating_pre / away_pi_rating_pre
  B) Context   : gebruikt home_pi_home_pre / away_pi_away_pre

Referenties
-----------
Corsten (2017). State Space Model for predicting match results in AFL.
Koopman & Lit (2015). Dynamic Bivariate Poisson Model for EPL.
Constantinou & Fenton (2013). Pi-ratings. JQAS.
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from scipy.stats import norm
import warnings
import json
from datetime import datetime

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_data(
    path: str = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/kalman_expected_goals.csv",
    pi_mode: str = "general",
) -> pd.DataFrame:
    """
    Laad data en selecteer pi-rating kolommen op basis van mode.

    pi_mode = "general" : home_pi_rating_pre / away_pi_rating_pre
                          (gemiddelde thuis/uit rating per team)
    pi_mode = "context" : home_pi_home_pre / away_pi_away_pre
                          (context-specifieke ratings)
    """
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    required = ["home_pi_rating_pre", "away_pi_rating_pre",
                "home_pi_home_pre",   "away_pi_away_pre",
                "lineup_strength_diff_norm"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Ontbrekende kolommen: {missing}\n"
            f"Draai eerst kalman_updater_with_pi.py."
        )

    # Selecteer pi-rating kolommen op basis van mode
    if pi_mode == "general":
        df["pi_home"] = df["home_pi_rating_pre"]
        df["pi_away"] = df["away_pi_rating_pre"]
    else:
        df["pi_home"] = df["home_pi_home_pre"]
        df["pi_away"] = df["away_pi_away_pre"]

    return df


# ─────────────────────────────────────────────────────────────────────────────
# 2.  KALMAN FILTER MET ASYMMETRISCHE PI-WEGING
# ─────────────────────────────────────────────────────────────────────────────

class KalmanXGFilter:
    """
    Bivariate Kalman filter met asymmetrische Pi-rating gewogen Kalman gain.

    De aanvals- en verdedigingsstates worden apart gewogen op basis van
    de absolute kwaliteit van de tegenstander (niet het verschil):

      w_att_home = f(away_pi)  — aanval thuisploeg: informatiever tegen sterkere uitploeg
      w_def_away = f(home_pi)  — verdediging uitploeg: informatiever als thuisploeg sterker is
      w_att_away = f(home_pi)  — aanval uitploeg: informatiever tegen sterkere thuisploeg
      w_def_home = f(away_pi)  — verdediging thuisploeg: informatiever als uitploeg sterker is

    Dit lost het identificatieprobleem deels op: γ ontvangt nu
    gedifferentieerde signalen die afhangen van de thuisploeg kwaliteit,
    wat de optimizer een reden geeft om σ²_w_def > 0 te houden.
    """

    def __init__(self, sigma2_w_att: float, sigma2_w_def: float,
                 sigma2_v: float, phi: float, eta: float,
                 lambda_att: float = 0.0, lambda_def: float = 0.0,
                 lambda_lineup: float = 0.0, home_adv: float = 0.15):
        self.sigma2_w_att  = max(sigma2_w_att, 1e-8)
        self.sigma2_w_def  = max(sigma2_w_def, 1e-8)
        self.sigma2_v      = max(sigma2_v, 1e-8)
        self.phi           = np.clip(phi, 0.01, 0.999)
        self.eta           = eta
        self.lambda_att    = lambda_att
        self.lambda_def    = lambda_def
        self.lambda_lineup = lambda_lineup
        self.home_adv      = home_adv
        self.states: dict  = {}

    def _init_team(self, team: str):
        if team not in self.states:
            self.states[team] = {
                "alpha": 0.0, "gamma": 0.0,
                "var_alpha": 1.0, "var_gamma": 1.0,
            }

    def _predict_step(self, team: str):
        s = self.states[team]
        return (
            self.phi * s["alpha"],
            self.phi * s["gamma"],
            self.phi**2 * s["var_alpha"] + self.sigma2_w_att,
            self.phi**2 * s["var_gamma"] + self.sigma2_w_def,
        )

    def _weights(self, pi_home: float, pi_away: float, ls_diff_norm: float):
        """
        Asymmetrische innovatiegewichten per state:

        Aanval thuisploeg (w_att_h):
          Hoog als uitploeg sterk is → 3 xG scoren tegen Man City
          zegt meer over Liverpool's aanval dan tegen Norwich

        Verdediging uitploeg (w_def_a):
          Hoog als thuisploeg sterk is → 3 xG toelaten van Liverpool
          zegt meer over Norwich's verdediging dan toelaten van Burnley

        Lineup gewicht is symmetrisch toegevoegd aan beide attack gewichten
        want sterkere opstelling beïnvloedt primair de aanvalskracht.
        """
        w_att_h = 1.0 + self.lambda_att * pi_away + self.lambda_lineup * (-ls_diff_norm)
        w_def_a = 1.0 + self.lambda_def * pi_home
        w_att_a = 1.0 + self.lambda_att * pi_home + self.lambda_lineup * ( ls_diff_norm)
        w_def_h = 1.0 + self.lambda_def * pi_away

        w_att_h = float(np.clip(w_att_h, 0.1, 3.0))
        w_def_a = float(np.clip(w_def_a, 0.1, 3.0))
        w_att_a = float(np.clip(w_att_a, 0.1, 3.0))
        w_def_h = float(np.clip(w_def_h, 0.1, 3.0))

        return w_att_h, w_def_a, w_att_a, w_def_h

    def predict_xg(self, home: str, away: str,
                   lineup_diff_norm: float,
                   pi_home: float = 0.0, pi_away: float = 0.0):
        self._init_team(home); self._init_team(away)
        ah, _, vah, _    = self._predict_step(home)
        aa, ga, vaa, vga = self._predict_step(away)
        _, gh, _, vgh    = self._predict_step(home)
        # EKF log-link: xG = exp(δ + α − γ + η·ls)
        xg_home = np.exp(np.clip(self.home_adv + ah - ga + self.eta * lineup_diff_norm, -10, 10))
        xg_away = np.exp(np.clip(               aa - gh - self.eta * lineup_diff_norm, -10, 10))
        # EKF innovatievariantie via Jacobiaan: S = xG² · (P_α + P_γ) + σ²_v
        S_home = xg_home**2 * (vah + vga) + self.sigma2_v
        S_away = xg_away**2 * (vaa + vgh) + self.sigma2_v
        return xg_home, xg_away, S_home, S_away

    def update(self, home: str, away: str,
               obs_xg_home: float, obs_xg_away: float,
               lineup_diff_norm: float,
               pi_home: float = 0.0, pi_away: float = 0.0):
        self._init_team(home); self._init_team(away)

        # Predictie stap
        ah_pred, gh_pred, vah_pred, vgh_pred = self._predict_step(home)
        aa_pred, ga_pred, vaa_pred, vga_pred = self._predict_step(away)

        # Innovaties (obs − pred in oorspronkelijke xG schaal)
        # xG_pred wordt berekend na Jacobiaan sectie hieronder
        e_home = obs_xg_home - np.exp(np.clip(self.home_adv + ah_pred - ga_pred + self.eta * lineup_diff_norm, -10, 10))
        e_away = obs_xg_away - np.exp(np.clip(                aa_pred - gh_pred - self.eta * lineup_diff_norm, -10, 10))

        # EKF log-link observatievergelijking
        xg_home_pred = np.exp(np.clip(self.home_adv + ah_pred - ga_pred + self.eta * lineup_diff_norm, -10, 10))
        xg_away_pred = np.exp(np.clip(                aa_pred - gh_pred - self.eta * lineup_diff_norm, -10, 10))

        # EKF innovatievariantie via Jacobiaan: S = xG_pred² · (P_α + P_γ) + σ²_v
        G_home = xg_home_pred**2 * (vah_pred + vga_pred) + self.sigma2_v
        G_away = xg_away_pred**2 * (vaa_pred + vgh_pred) + self.sigma2_v

        # Log-likelihood bijdrage
        ll = (
            -0.5 * np.log(2 * np.pi * G_home) - 0.5 * e_home**2 / G_home
            - 0.5 * np.log(2 * np.pi * G_away) - 0.5 * e_away**2 / G_away
        )

        # EKF Kalman gains (geschaald met xG_pred via Jacobiaan)
        K_ah = xg_home_pred * vah_pred / G_home
        K_ga = xg_home_pred * vga_pred / G_home
        K_aa = xg_away_pred * vaa_pred / G_away
        K_gh = xg_away_pred * vgh_pred / G_away

        # Asymmetrische innovatiegewichten
        w_att_h, w_def_a, w_att_a, w_def_h = self._weights(
            pi_home, pi_away, lineup_diff_norm
        )

        # State updates — aanval en verdediging krijgen eigen gewicht
        self.states[home]["alpha"] = ah_pred + K_ah * e_home * w_att_h
        self.states[away]["gamma"] = ga_pred - K_ga * e_home * w_def_a
        self.states[away]["alpha"] = aa_pred + K_aa * e_away * w_att_a
        self.states[home]["gamma"] = gh_pred - K_gh * e_away * w_def_h

        # Variantie updates (EKF Joseph form)
        self.states[home]["var_alpha"] = max((1 - K_ah * xg_home_pred) * vah_pred, 1e-8)
        self.states[home]["var_gamma"] = max((1 - K_gh * xg_away_pred) * vgh_pred, 1e-8)
        self.states[away]["var_alpha"] = max((1 - K_aa * xg_away_pred) * vaa_pred, 1e-8)
        self.states[away]["var_gamma"] = max((1 - K_ga * xg_home_pred) * vga_pred, 1e-8)

        return ll

    def run_on_sequence(self, df_seq: pd.DataFrame) -> float:
        total_ll = 0.0
        for _, row in df_seq.iterrows():
            ll = self.update(
                home=row["home_team"], away=row["away_team"],
                obs_xg_home=row["home_xg"], obs_xg_away=row["away_xg"],
                lineup_diff_norm=row["lineup_strength_diff_norm"],
                pi_home=row.get("pi_home", 0.0),
                pi_away=row.get("pi_away", 0.0),
            )
            if np.isfinite(ll):
                total_ll += ll
        return total_ll


# ─────────────────────────────────────────────────────────────────────────────
# 3.  METRICS
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_predictions(df_eval: pd.DataFrame, kf: KalmanXGFilter) -> dict:
    ph, pa, sh, sa = [], [], [], []
    ah = df_eval["home_xg"].values
    aa = df_eval["away_xg"].values

    for _, row in df_eval.iterrows():
        xgh, xga, Gh, Ga = kf.predict_xg(
            row["home_team"], row["away_team"],
            row["lineup_strength_diff_norm"],
            pi_home=row.get("pi_home", 0.0),
            pi_away=row.get("pi_away", 0.0),
        )
        ph.append(xgh); pa.append(xga)
        sh.append(np.sqrt(Gh)); sa.append(np.sqrt(Ga))

    ph, pa = np.array(ph), np.array(pa)
    sh, sa = np.array(sh), np.array(sa)

    def mae(p, a):    return np.mean(np.abs(p - a))
    def rmse(p, a):   return np.sqrt(np.mean((p - a)**2))
    def nll(p, s, a): return -np.mean(norm.logpdf(a, loc=p, scale=np.maximum(s, 1e-6)))

    return {
        "mae_home":   mae(ph, ah),  "mae_away":   mae(pa, aa),
        "rmse_home":  rmse(ph, ah), "rmse_away":  rmse(pa, aa),
        "nll_home":   nll(ph, sh, ah), "nll_away": nll(pa, sa, aa),
        "mae_total":  mae(np.r_[ph, pa], np.r_[ah, aa]),
        "rmse_total": rmse(np.r_[ph, pa], np.r_[ah, aa]),
        "nll_total":  0.5 * (nll(ph, sh, ah) + nll(pa, sa, aa)),
        "n_obs": len(df_eval),
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4.  OBJECTIVE
# ─────────────────────────────────────────────────────────────────────────────

def neg_log_likelihood(params: np.ndarray, df_train: pd.DataFrame) -> float:
    """
    params = [log(σ²_w_att), log(σ²_w_def), log(σ²_v),
              φ_raw, η, λ_att, λ_def, λ_lineup, home_adv]
    """
    log_sw_att, log_sw_def, log_sv, phi_raw, eta, lam_att, lam_def, lam_lineup, home_adv = params
    kf = KalmanXGFilter(
        sigma2_w_att=np.exp(log_sw_att),
        sigma2_w_def=np.exp(log_sw_def),
        sigma2_v    =np.exp(log_sv),
        phi         =1.0 / (1.0 + np.exp(-phi_raw)),
        eta=eta,
        lambda_att=lam_att,
        lambda_def=lam_def,
        lambda_lineup=lam_lineup,
        home_adv=home_adv,
    )
    ll = kf.run_on_sequence(df_train)
    return -ll if np.isfinite(ll) else 1e10


# ─────────────────────────────────────────────────────────────────────────────
# 5.  WALK-FORWARD VALIDATOR
# ─────────────────────────────────────────────────────────────────────────────

class WalkForwardValidator:

    PARAM_COLS = ["sigma2_w_att", "sigma2_w_def", "sigma2_v",
                  "phi", "eta", "lambda_att", "lambda_def", "lambda_lineup", "home_adv"]

    def __init__(self, df, min_train_seasons=2, optimiser="two_stage"):
        self.df                = df.copy()
        self.seasons           = sorted(df["season"].unique())
        self.min_train_seasons = min_train_seasons
        self.optimiser         = optimiser
        self.fold_results      = []
        self.optimal_params_per_fold = []

    @staticmethod
    def _bounds():
        return [
            (-11.5, 0.69),  # log σ²_w_att
            (-11.5, 0.69),  # log σ²_w_def
            (-11.5, 1.61),  # log σ²_v
            ( 0.00, 6.90),  # φ_raw (sigmoid)
            (-1.00, 1.00),  # η
            ( 0.00, 2.00),  # λ_att  — positief: sterkere tegenstander = meer info voor aanval
            ( 0.00, 2.00),  # λ_def  — positief: sterkere tegenstander = meer info voor verdediging
            (-1.00, 1.00),  # λ_lineup
            ( 0.00, 0.50),  # home_adv (thuisvoordeel in xG)
        ]

    @staticmethod
    def _x0():
        return [np.log(0.05), np.log(0.05), np.log(0.30),
                4.60, 0.05, 0.10, 0.10, 0.05, 0.15]

    @staticmethod
    def _decode(params):
        log_sa, log_sd, log_sv, phi_r, eta, latt, ldef, llu, hadv = params
        return {
            "sigma2_w_att": float(np.exp(log_sa)),
            "sigma2_w_def": float(np.exp(log_sd)),
            "sigma2_v"    : float(np.exp(log_sv)),
            "phi"         : float(1.0 / (1.0 + np.exp(-phi_r))),
            "eta"         : float(eta),
            "lambda_att"  : float(latt),
            "lambda_def"  : float(ldef),
            "lambda_lineup": float(llu),
            "home_adv"    : float(hadv),
        }

    def _optimise(self, df_train):
        bounds = self._bounds()
        if self.optimiser == "two_stage":
            de = differential_evolution(
                neg_log_likelihood, bounds, args=(df_train,),
                strategy="best1bin", maxiter=300, popsize=12,
                tol=1e-6, mutation=(0.5, 1.5), recombination=0.8,
                seed=42, polish=False, disp=False,
            )
            res = minimize(neg_log_likelihood, de.x, args=(df_train,),
                           method="L-BFGS-B", bounds=bounds,
                           options={"maxiter": 1000, "ftol": 1e-10})
            return res.x
        else:
            res = minimize(neg_log_likelihood, self._x0(), args=(df_train,),
                           method="L-BFGS-B", bounds=bounds,
                           options={"maxiter": 1000, "ftol": 1e-10})
            return res.x

    def run(self, verbose=True) -> pd.DataFrame:
        fold_records = []
        folds = list(range(self.min_train_seasons, len(self.seasons)))

        for fold_idx, val_idx in enumerate(folds):
            train_seasons = self.seasons[:val_idx]
            val_season    = self.seasons[val_idx]
            df_train = self.df[self.df["season"].isin(train_seasons)].copy()
            df_val   = self.df[self.df["season"] == val_season].copy()

            if verbose:
                print(f"\n{'='*65}")
                print(f"  Fold {fold_idx+1}/{len(folds)}")
                print(f"  Train: {train_seasons[0]} → {train_seasons[-1]}"
                      f"  ({len(df_train)} matches)")
                print(f"  Valid: {val_season}  ({len(df_val)} matches)")
                print(f"{'='*65}")
                print(f"  Optimising θ via '{self.optimiser}'...")

            t0      = datetime.now()
            raw     = self._optimise(df_train)
            theta   = self._decode(raw)
            elapsed = (datetime.now() - t0).total_seconds()

            if verbose:
                print(f"  Completed in {elapsed:.1f}s")
                print(f"  θ* = σ²_w_att={theta['sigma2_w_att']:.5f}, "
                      f"σ²_w_def={theta['sigma2_w_def']:.5f}, "
                      f"σ²_v={theta['sigma2_v']:.5f}, φ={theta['phi']:.4f}")
                print(f"       η={theta['eta']:.4f}, "
                      f"λ_att={theta['lambda_att']:.4f}, "
                      f"λ_def={theta['lambda_def']:.4f}, "
                      f"λ_lineup={theta['lambda_lineup']:.4f}, "
                      f"home_adv={theta['home_adv']:.4f}")

            kf       = KalmanXGFilter(**theta)
            train_ll = kf.run_on_sequence(df_train)
            metrics  = evaluate_predictions(df_val, kf)

            fold_records.append({
                "fold": fold_idx + 1,
                "train_seasons": " | ".join(train_seasons),
                "val_season": val_season,
                "n_train": len(df_train),
                "n_val": len(df_val),
                "train_ll": train_ll,
                "opt_time_s": elapsed,
                **theta,
                **{f"val_{k}": v for k, v in metrics.items()},
            })
            self.optimal_params_per_fold.append(theta)

            if verbose:
                print(f"  Out-of-sample → MAE={metrics['mae_total']:.4f}, "
                      f"RMSE={metrics['rmse_total']:.4f}, "
                      f"NLL={metrics['nll_total']:.4f}")

        self.fold_results = pd.DataFrame(fold_records)
        return self.fold_results


# ─────────────────────────────────────────────────────────────────────────────
# 6.  VERGELIJKING: ALGEMEEN vs CONTEXT-SPECIFIEK
# ─────────────────────────────────────────────────────────────────────────────

def compare_pi_modes(
    path: str = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/kalman_expected_goals.csv",
    min_train_seasons: int = 2,
) -> pd.DataFrame:
    """
    Vergelijk algemene vs context-specifieke Pi-rating specificatie.
    Beide gebruiken de asymmetrische λ_att / λ_def weging.
    """
    comparison_records = []

    for mode, label in [("general", "Algemeen (gem. thuis/uit)"),
                        ("context", "Context-specifiek (thuis vs uit)")]:
        print(f"\n{'='*65}")
        print(f"  PI-RATING MODE: {label}")
        print(f"{'='*65}")

        df  = load_data(path, pi_mode=mode)
        wfv = WalkForwardValidator(df, min_train_seasons=min_train_seasons,
                                   optimiser="two_stage")
        results = wfv.run(verbose=True)

        for _, row in results.iterrows():
            comparison_records.append({
                "pi_mode"     : label,
                "fold"        : row["fold"],
                "val_season"  : row["val_season"],
                "mae_total"   : row["val_mae_total"],
                "rmse_total"  : row["val_rmse_total"],
                "nll_total"   : row["val_nll_total"],
                "lambda_att"  : row["lambda_att"],
                "lambda_def"  : row["lambda_def"],
                "sigma2_w_def": row["sigma2_w_def"],
            })

    comp_df = pd.DataFrame(comparison_records)

    print("\n\n" + "="*65)
    print("  VERGELIJKING ALGEMEEN vs CONTEXT-SPECIFIEK")
    print("="*65)
    summary = comp_df.groupby("pi_mode")[
        ["mae_total","rmse_total","nll_total","lambda_att","lambda_def","sigma2_w_def"]
    ].agg(["mean","std"])
    print(summary.to_string(float_format=lambda x: f"{x:.5f}"))

    gen_mae = comp_df[comp_df["pi_mode"].str.startswith("Alg")]["mae_total"].mean()
    ctx_mae = comp_df[comp_df["pi_mode"].str.startswith("Con")]["mae_total"].mean()
    print(f"\n  Algemeen MAE  : {gen_mae:.4f}")
    print(f"  Context  MAE  : {ctx_mae:.4f}")
    print(f"  Beste spec.   : {'Algemeen' if gen_mae < ctx_mae else 'Context-specifiek'}")

    return comp_df


# ─────────────────────────────────────────────────────────────────────────────
# 7.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("\n" + "="*65)
    print("  KALMAN xG — ASYMMETRISCHE PI-RATING GEWOGEN KALMAN GAIN")
    print("  θ = (σ²_w_att, σ²_w_def, σ²_v, φ, η, λ_att, λ_def, λ_lineup, home_adv)")
    print("="*65)

    DATA_PATH = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/kalman_expected_goals.csv"
    OUT_PATH  = r"C:/Users/semwi/FPL-Core-Insights/data/Kalman data/"

    # Stap 1: Hoofdanalyse — algemene pi-rating
    print("\n[STAP 1] Hoofdanalyse — algemene pi-rating")
    df      = load_data(DATA_PATH, pi_mode="general")
    print(f"Loaded {len(df)} matches, {df['season'].nunique()} seasons")

    wfv     = WalkForwardValidator(df, min_train_seasons=2, optimiser="two_stage")
    results = wfv.run(verbose=True)

    pc = WalkForwardValidator.PARAM_COLS
    mc = ["val_mae_total", "val_rmse_total", "val_nll_total"]

    print("\n\n" + "="*65)
    print("  RESULTATEN PER FOLD")
    print("="*65)
    print(results[["fold","val_season","n_train","n_val"] + pc + mc].to_string(
          index=False, float_format=lambda x: f"{x:.5f}"))

    print("\n\n" + "="*65)
    print("  AGGREGATE OVER ALLE FOLDS")
    print("="*65)
    print(results[mc].agg(["mean","std","min","max"]).to_string(
          float_format=lambda x: f"{x:.5f}"))

    print("\n\n" + "="*65)
    print("  HYPERPARAMETER STABILITEIT — let op σ²_w_def vs vorige versie")
    print("="*65)
    print(results[pc].agg(["mean","std","min","max"]).to_string(
          float_format=lambda x: f"{x:.6f}"))

    # Opslaan
    results.to_csv(f"{OUT_PATH}wfv_fold_results.csv", index=False)

    with open(f"{OUT_PATH}optimal_hyperparameters.json", "w") as f:
        json.dump({
            "pi_mode"   : "general",
            "mean"      : {p: float(results[p].mean()) for p in pc},
            "final_fold": {p: float(results.iloc[-1][p]) for p in pc},
            "per_fold"  : [{p: float(results.iloc[i][p]) for p in pc}
                           for i in range(len(results))],
        }, f, indent=2)

    print(f"\nOpgeslagen:")
    print(f"  {OUT_PATH}wfv_fold_results.csv")
    print(f"  {OUT_PATH}optimal_hyperparameters.json")

    return results


if __name__ == "__main__":
    results = main()


  KALMAN xG — ASYMMETRISCHE PI-RATING GEWOGEN KALMAN GAIN
  θ = (σ²_w_att, σ²_w_def, σ²_v, φ, η, λ_att, λ_def, λ_lineup, home_adv)

[STAP 1] Hoofdanalyse — algemene pi-rating
Loaded 2608 matches, 7 seasons

  Fold 1/5
  Train: 2019/2020 → 2020/2021  (760 matches)
  Valid: 2021/2022  (380 matches)
  Optimising θ via 'two_stage'...
  Completed in 1965.9s
  θ* = σ²_w_att=0.01538, σ²_w_def=0.01746, σ²_v=0.46843, φ=0.8223
       η=0.2608, λ_att=0.0161, λ_def=0.0000, λ_lineup=0.1941, home_adv=0.1771
  Out-of-sample → MAE=0.6035, RMSE=0.7816, NLL=1.1686

  Fold 2/5
  Train: 2019/2020 → 2021/2022  (1140 matches)
  Valid: 2022/2023  (380 matches)
  Optimising θ via 'two_stage'...
  Completed in 2812.0s
  θ* = σ²_w_att=0.00870, σ²_w_def=0.00890, σ²_v=0.47553, φ=0.8921
       η=0.2689, λ_att=0.1686, λ_def=0.0000, λ_lineup=0.2183, home_adv=0.1870
  Out-of-sample → MAE=0.6310, RMSE=0.8220, NLL=1.2441

  Fold 3/5
  Train: 2019/2020 → 2022/2023  (1520 matches)
  Valid: 2023/2024  (380 matches)
  Opt